In [2]:
import cv2
import torch
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf
from collections import defaultdict, Counter
from ultralytics import YOLO
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "MAIN_MODULE" / "src"))
sys.path.insert(0, str(Path.cwd() / "DEPARTMENT_CLASSIFICATION" / "train_model"))
sys.path.insert(0, str(Path.cwd() / "IMG PREPROCESSING"))
sys.path.insert(0, str(Path.cwd() / "CROP_QUALITY_CLASSIFICATION"))

from quality_classifier.predict import quality_classifier
from color_classifier import process_dataset
from crop_extraction import CropCandidate, CropScorer
from predict_single import DepartmentPredictor



sys.path.insert(0, str(Path.cwd() / "VLM_MODULE"))
from detect import load_vlm_model, vlm_predict_crops
sys.path.insert(0, str(Path.cwd() / "LLMTEXT"))
from LLMTEXT.hf_sku_matcher import HFSKUMatcher
from product_matcher import find_top5_matches

c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
config = OmegaConf.load('params.yaml')
root = Path.cwd()
video_folder = root / config.main_extraction.input_folder
yolo_path = root / config.main_extraction.model_path
dept_model_path = root / config.department_classifier.model_path
class_names_path = root / "DEPARTMENT_CLASSIFICATION/train_model/models/class_names.json"

In [4]:
video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".webm"}
videos = sorted(p for p in video_folder.iterdir()
                if p.suffix.lower() in video_extensions and not p.name.startswith("~"))
video_path = videos[0]
print(f"Видео: {video_path.name}")

Видео: 25_12-20.mp4


In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo = YOLO(str(yolo_path)).to(device)
crop_scorer = CropScorer(
    config.main_extraction.min_crop_width,
    config.main_extraction.min_crop_height,
    config.main_extraction.sharpness_threshold,
)

def _predict_np(self, img, top_k=3):
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    elif img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = self.transform(image=img)
    x = t['image'].unsqueeze(0).to(self.device)
    with torch.no_grad():
        p = torch.softmax(self.model(x), dim=1)[0]
    top = torch.topk(p, top_k)
    return [(self.class_names[i.item()], v.item() * 100) for v, i in zip(top.values, top.indices)]

DepartmentPredictor.predict_np = _predict_np
classifier = DepartmentPredictor(str(dept_model_path), str(class_names_path))

Создана efficientnet-b0 со случайными весами
Модель загружена: efficientnet-b0
Классов: 15
Устройство: cuda


In [6]:
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"{total} frames, {fps:.2f} FPS")

865 frames, 19.96 FPS


In [7]:
seg_size = total // 5
half_window = 15
step = 2
segment_frames = []
for i in range(5):
    mid = i * seg_size + seg_size // 2
    start = max(0, mid - half_window)
    end = min(total - 1, mid + half_window)
    segment_frames.append(list(range(start, end + 1, step)))
for i, frames in enumerate(segment_frames):
    print(f"  Сегмент {i+1}: {len(frames)} кадров ({frames[0]/fps:.1f}с - {frames[-1]/fps:.1f}с)")

  Сегмент 1: 16 кадров (3.6с - 5.1с)
  Сегмент 2: 16 кадров (12.2с - 13.7с)
  Сегмент 3: 16 кадров (20.9с - 22.4с)
  Сегмент 4: 16 кадров (29.6с - 31.1с)
  Сегмент 5: 16 кадров (38.2с - 39.7с)


In [8]:
best: dict[int, list[CropCandidate]] = defaultdict(list)
frame_to_seg = {}
for sid, frames in enumerate(segment_frames):
    for f in frames:
        frame_to_seg[f] = sid
seg_preds = [[] for _ in range(5)]

cap = cv2.VideoCapture(str(video_path))
fi = -1
while True:
    ok, fr = cap.read()
    if not ok:
        break
    fi += 1

    if config.main_extraction.rotate_frames:
        fr = cv2.rotate(fr, cv2.ROTATE_90_COUNTERCLOCKWISE)

    res = yolo.track(source=fr, persist=True, tracker=config.main_extraction.tracker_config,
                     conf=config.main_extraction.conf_threshold, iou=config.main_extraction.iou_threshold, verbose=False)[0]

    boxes = None
    if res.boxes is not None and res.boxes.id is not None:
        boxes = res.boxes.xyxy.cpu().numpy()
        for box, conf, tid in zip(boxes, res.boxes.conf.cpu().numpy(), res.boxes.id.cpu().numpy().astype(int)):
            x1, y1, x2, y2 = map(int, box)
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(fr.shape[1], x2), min(fr.shape[0], y2)
            crop = fr[y1:y2, x1:x2]
            score = crop_scorer.compute_score(crop, conf)
            if score is None:
                continue
            c = CropCandidate(score, crop.copy(), fi, float(conf), [x1, y1, x2, y2])
            best[tid].append(c)
            best[tid].sort(key=lambda x: x.score, reverse=True)
            best[tid] = best[tid][:config.main_extraction.top_k]

    if fi in frame_to_seg:
        sid = frame_to_seg[fi]
        dept, prob = classifier.predict_np(fr)[0]
        seg_preds[sid].append({'frame': fi, 'time': fi / fps, 'department': dept, 'prob': prob})

    if fi % 500 == 0:
        print(f"Frame {fi}/{total}, tracks: {len(best)}")

cap.release()
print(f"Done. Tracks: {len(best)}, crops: {sum(len(v) for v in best.values())}")

Frame 0/865, tracks: 8
Frame 500/865, tracks: 42
Done. Tracks: 68, crops: 68


In [9]:
# Сбор результатов по предсказанию отдела
rows_seg = []
for sid, preds in enumerate(seg_preds):
    for p in preds:
        rows_seg.append({
            'segment': sid + 1,
            'frame': p['frame'],
            'time_sec': round(p['time'], 1),
            'department': p['department'],
            'probability': round(p['prob'], 1),
        })

department = pd.DataFrame(rows_seg)

In [10]:
department

,segment,frame,time_sec,department,probability
0,1,71,3.6,напитки снэки,87.4
1,1,73,3.7,напитки снэки,80.9
2,1,75,3.8,напитки снэки,75.2
3,1,77,3.9,напитки снэки,66.3
4,1,79,4.0,Детские товары,48.7
...,...,...,...,...,...
75,5,785,39.3,напитки снэки,86.9
76,5,787,39.4,напитки снэки,85.7
77,5,789,39.5,напитки снэки,93.0
78,5,791,39.6,напитки снэки,96.8


In [11]:
rows_crops = []
for track_id, candidates in best.items():
    for rank, c in enumerate(candidates):
        rows_crops.append({
            'filename': video_path.name,
            'SYS_track_id': track_id,
            'SYS_rank': rank + 1,
            'SYS_score': round(c.score, 1),
            'SYS_confidence': round(c.confidence, 3),
            'product_name': None, 'price_default': None, 'price_card': None,
            'price_discount': None, 'barcode': None, 'discount_amount': None,
            'id_sku': None, 'print_datetime': None, 'code': None,
            'additional_info': None, 'color': None, 'special_symbols': None,
            'frame_timestamp': int(c.frame_index / fps * 1000),
            'x_min': c.bbox[0], 'y_min': c.bbox[1],
            'x_max': c.bbox[2], 'y_max': c.bbox[3],
            'qr_code_barcode': None, 'price1_qr': None, 'price2_qr': None,
            'price3_qr': None, 'price4_qr': None,
            'wholesale_level_1_count': None, 'wholesale_level_1_price': None,
            'wholesale_level_2_count': None, 'wholesale_level_2_price': None,
            'action_price_qr': None, 'action_code_qr': None,
        })

df_crops = pd.DataFrame(rows_crops)

In [12]:
crops_for_color = [(track_id, candidate.crop) 
                   for track_id, candidates in best.items() 
                   for candidate in candidates]
# Классификация цветов
color_results = process_dataset(crops_for_color)
# Добавление в DataFrame
df_crops['color'] = df_crops['SYS_track_id'].map(color_results)

In [13]:
# Сбор кропов для классификации качества
crops_for_quality = [(track_id, candidate.crop) 
                     for track_id, candidates in best.items() 
                     for candidate in candidates]

# Классификация качества (мусор/нет)
quality_model_path = root / config.quality_classifier.model_path
trash_map, confidence_map = quality_classifier(str(quality_model_path), crops_for_quality)

# Добавление в DataFrame
df_crops['SYS_trash'] = df_crops['SYS_track_id'].map(trash_map)


Создана MobileNetV3 со случайными весами


In [14]:

# # === ВИЗУАЛИЗАЦИЯ ДЛЯ ПРОВЕРКИ МУСОРА ===
# df_check = pd.DataFrame([
#     {
#         'track_id': track_id,
#         'is_trash': trash_map[track_id],
#         'confidence': confidence_map[track_id],
#         'crop': crop_image
#     }
#     for track_id, crop_image in crops_for_quality
# ])

# # Отдельно плохие и хорошие
# df_bad = df_check[df_check['is_trash'] == True].sort_values('confidence', ascending=False)
# df_good = df_check[df_check['is_trash'] == False].sort_values('confidence', ascending=True)

# # Показать ВСЕ примеры
# n_bad, n_good = len(df_bad), len(df_good)
# n_total = max(n_bad, n_good)

# if n_total > 0:
#     if n_total == 1:
#         fig, axes = plt.subplots(1, 2, figsize=(12, 6))
#         axes = np.array([[axes[0], axes[1]]])
#     else:
#         fig, axes = plt.subplots(n_total, 2, figsize=(12, 6 * n_total))
#         if n_total == 1:
#             axes = axes.reshape(1, -1)
    
#     for i, (_, row) in enumerate(df_bad.iterrows()):
#         axes[i, 0].imshow(cv2.cvtColor(row['crop'], cv2.COLOR_BGR2RGB))
#         axes[i, 0].set_title(f"Мусор: {row['confidence']:.1f}%")
#         axes[i, 0].axis('off')
    
#     for i, (_, row) in enumerate(df_good.iterrows()):
#         axes[i, 1].imshow(cv2.cvtColor(row['crop'], cv2.COLOR_BGR2RGB))
#         axes[i, 1].set_title(f"Хороший: {100 - row['confidence']:.1f}%")
#         axes[i, 1].axis('off')
    
#     # Скрыть пустые ячейки
#     for i in range(n_good, n_total):
#         axes[i, 1].axis('off')
#     for i in range(n_bad, n_total):
#         axes[i, 0].axis('off')
    
#     plt.tight_layout()
#     plt.show()

# print(f"Мусор: {len(df_bad)}, Хороших: {len(df_good)}")

In [15]:
# === ТРАНСФОРМАЦИЯ КООРДИНАТ ===
# Координаты сейчас записаны для кадров, повернутых на 90° против часовой
# Оригинальное видео: 3840x2160 (WxH)
# После поворота: 2160x3840
# Формула преобразования (поворот на 90° по часовой):
# x_min_orig = W_orig - y_max_rot
# y_min_orig = x_min_rot
# x_max_orig = W_orig - y_min_rot
# y_max_orig = x_max_rot

W_orig, H_orig = 3840, 2160

def transform_coords(row):
    x_min_rot = row['x_min']
    y_min_rot = row['y_min']
    x_max_rot = row['x_max']
    y_max_rot = row['y_max']
    
    x_min_orig = W_orig - y_max_rot
    y_min_orig = x_min_rot
    x_max_orig = W_orig - y_min_rot
    y_max_orig = x_max_rot
    
    return pd.Series({
        'x_min_orig': int(x_min_orig),
        'y_min_orig': int(y_min_orig),
        'x_max_orig': int(x_max_orig),
        'y_max_orig': int(y_max_orig)
    })

df_crops[['x_min_orig', 'y_min_orig', 'x_max_orig', 'y_max_orig']] = df_crops.apply(transform_coords, axis=1)

In [16]:
# # === ВИЗУАЛИЗАЦИЯ ПРОВЕРКИ 10 оригинальных кадров ===
# import cv2
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches

# # Берем первые 10 строк df_crops
# top_10 = df_crops.head(10).copy()

# # Группируем по frame_timestamp для определения уникальных кадров
# unique_timestamps = top_10['frame_timestamp'].unique()

# # Параметры видео
# video_path_str = str(root / 'data' / '25_12-20' / '25_12-20.mp4')
# cap = cv2.VideoCapture(video_path_str)
# fps = cap.get(cv2.CAP_PROP_FPS)
# total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# cap.release()

# # Функция для получения кадра по timestamp (в мс)
# def get_frame_by_timestamp(video_path, timestamp_ms):
#     cap = cv2.VideoCapture(video_path)
#     frame_num = int(timestamp_ms / 1000 * fps)
#     cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
#     ok, frame = cap.read()
#     cap.release()
#     if ok:
#         return frame, frame_num
#     else:
#         return None, None

# # Собираем данные для визуализации
# frames_data = {}
# for ts in unique_timestamps:
#     frame, frame_num = get_frame_by_timestamp(video_path_str, ts)
#     if frame is not None:
#         frames_data[ts] = {
#             'frame': frame,
#             'frame_num': frame_num,
#             'crops': top_10[top_10['frame_timestamp'] == ts]
#         }

# # Визуализация
# n_frames = len(frames_data)
# if n_frames > 0:
#     fig, axes = plt.subplots(n_frames, 1, figsize=(16, 9 * n_frames))
#     if n_frames == 1:
#         axes = [axes]
    
#     for idx, (ts, data) in enumerate(frames_data.items()):
#         ax = axes[idx]
#         frame_rgb = cv2.cvtColor(data['frame'], cv2.COLOR_BGR2RGB)
#         ax.imshow(frame_rgb)
        
#         # Наложение bbox
#         for _, crop_row in data['crops'].iterrows():
#             x_min = crop_row['x_min_orig']
#             y_min = crop_row['y_min_orig']
#             x_max = crop_row['x_max_orig']
#             y_max = crop_row['y_max_orig']
            
#             rect = patches.Rectangle(
#                 (x_min, y_min),
#                 x_max - x_min,
#                 y_max - y_min,
#                 linewidth=3,
#                 edgecolor='red' if crop_row['SYS_trash'] else 'green',
#                 facecolor='none',
#                 label=f"Track {crop_row['SYS_track_id']}"
#             )
#             ax.add_patch(rect)
            
#             # Подпись
#             ax.text(
#                 x_min, y_min - 10,
#                 f"Track:{crop_row['SYS_track_id']} Score:{crop_row['SYS_score']:.1f}",
#                 color='red' if crop_row['SYS_trash'] else 'green',
#                 fontsize=12,
#                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
#             )
        
#         ax.set_title(f"Frame {data['frame_num']} (Time: {ts}ms)")
#         ax.axis('off')
    
#     plt.tight_layout()
#     plt.show()

# print(f"Показано {n_frames} уникальных кадров")
# print(f"Всего кропов: {len(top_10)}")

In [17]:
import gc, torch
del yolo, classifier
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [18]:
df_crops[df_crops['SYS_trash'] == False]

,filename,SYS_track_id,SYS_rank,SYS_score,SYS_confidence,product_name,price_default,price_card,price_discount,barcode,...,wholesale_level_1_price,wholesale_level_2_count,wholesale_level_2_price,action_price_qr,action_code_qr,SYS_trash,x_min_orig,y_min_orig,x_max_orig,y_max_orig
0,25_12-20.mp4,1,1,2405.2,0.982,None,None,None,None,None,...,None,None,None,None,None,False,797,1614,978,1805
1,25_12-20.mp4,2,1,2635.5,0.983,None,None,None,None,None,...,None,None,None,None,None,False,776,1133,949,1345
2,25_12-20.mp4,3,1,2818.9,0.976,None,None,None,None,None,...,None,None,None,None,None,False,831,1676,1004,1859
3,25_12-20.mp4,4,1,3893.0,0.977,None,None,None,None,None,...,None,None,None,None,None,False,2029,1808,2223,1991
12,25_12-20.mp4,23,1,6213.4,0.983,None,None,None,None,None,...,None,None,None,None,None,False,755,1180,933,1386
16,25_12-20.mp4,34,1,4198.4,0.455,None,None,None,None,None,...,None,None,None,None,None,False,839,1938,1009,2121
17,25_12-20.mp4,39,1,7005.7,0.983,None,None,None,None,None,...,None,None,None,None,None,False,2081,201,2290,436
18,25_12-20.mp4,40,1,4321.4,0.264,None,None,None,None,None,...,None,None,None,None,None,False,769,1559,957,1906
20,25_12-20.mp4,54,1,5752.7,0.970,None,None,None,None,None,...,None,None,None,None,None,False,704,1022,881,1235
22,25_12-20.mp4,55,1,5146.1,0.983,None,None,None,None,None,...,None,None,None,None,None,False,701,763,875,980


In [19]:
# # Загрузка 
# vlm_model, vlm_processor = load_vlm_model(str(root / 'VLM_MODULE' / 'AVITO'), config_name='High Quality 8-bit')

# # crop_array from best into df_crops
# crop_map = {}
# for tid, candidates in best.items():
#     for rank, c in enumerate(candidates):
#         crop_map[(tid, rank + 1)] = c.crop

# df_crops['crop_array'] = df_crops.apply(
#     lambda r: crop_map.get((r['SYS_track_id'], r['SYS_rank']), None), axis=1)

# df_vlm = vlm_predict_crops(df_crops, vlm_model, vlm_processor)

In [20]:
df_vlm = pd.read_excel('vlm_new2.xlsx') 

In [22]:
df_vlm

,filename,SYS_track_id,SYS_rank,SYS_score,SYS_confidence,product_name,price_default,price_card,price_discount,barcode,...,action_price_qr,action_code_qr,price_without_card,price_with_card,promo_price,discount_size,article,layout_code,print_date,raw_text
0,25_12-20.mp4,1,1,2338.4,0.979,<0x0A>GRILL▁WINE▁<0xF0><0x9F><0x8D><0x87><0xF0...,NaN,NaN,NaN,12345_678,...,NaN,NaN,295₽,295₽,-26%,<0xE2><0x9C><0x8A>,12345_678,00:00,C123456789012345678901234567890123456789012345...,<0x0A>GRILL▁WINE▁<0xF0><0x9F><0x8D><0x87><0xF0...
1,25_12-20.mp4,2,1,3899.5,0.976,<0x0A>Вино▁ТОРО▁Бланко▁ординар▁белый▁сухие▁(Ар...,NaN,NaN,NaN,12345_678,...,NaN,NaN,659,659,-21%,<0xE2><0xAC><0x85>,12345_678,00▁2023-01-01,NaN,<0x0A>Вино▁ТОРО▁Бланко▁ординар▁белый▁сухие▁(Ар...
2,25_12-20.mp4,3,1,2799.6,0.970,<0x0A>ТОРО▁БИОЛОГИЧЕСКОЕ▁ОРГАНИЧЕСКОЕ▁БЕЗ▁ГМО▁...,NaN,NaN,NaN,1000_1000,...,NaN,NaN,659₽,659₽,-21%,00,1000_1000,1000_1000,1000_1000,<0x0A>ТОРО▁БИОЛОГИЧЕСКОЕ▁ОРГАНИЧЕСКОЕ▁БЕЗ▁ГМО▁...
3,25_12-20.mp4,4,1,2982.8,0.321,<0x0A>Продукция_название,NaN,NaN,NaN,Штрихкод,...,NaN,NaN,Цена_без_карты,Цена_с_картой,Прайм_цена,Размер_скидаки,Код_артикул,Локация_кода,Дата_печати,<0x0A>Продукция_название<>Цена_без_карты<>Цена...
4,25_12-20.mp4,5,1,5577.0,0.869,34▁15.09.2023<0x0A><0xF0><0x9F><0x94><0x84><0x...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,26%,12345_678,NaN,NaN,<0x0A>Вино▁GUSTARE▁<0x0A>1099₽<0x0A>1099₽<0x0A...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,25_12-20.mp4,255,1,2340.2,0.870,<0x0A>Продукция_название,NaN,NaN,NaN,Штрихкод,...,NaN,NaN,Цена_без_карты,Цена_с_картой,Прайм_цена,Размер_скидаки,Код_артикул,Формат_кода,"Дата_печати<0x0A><0x0A>На▁изображении▁видно,▁ч...",<0x0A>Продукция_название<>Цена_без_карты<>Цена...
59,25_12-20.mp4,257,1,1414.9,0.924,<0x0A>Продукция_название,NaN,NaN,NaN,Штрихкод,...,NaN,NaN,Цена_без_карты,Цена_с_картой,Скидка_размер,Вид_скидки,Код_артикул,Формат_кода,Дата_печати<0x0A><0x0A>Внимательно▁изучите▁изо...,<0x0A>Продукция_название<>Цена_без_карты<>Цена...
60,25_12-20.mp4,258,1,970.1,0.512,<0x0A>Продукция_название,NaN,NaN,NaN,Штрихкод,...,NaN,NaN,Цена_без_карты,Цена_с_картой,Скидка_размер,Вид_скидки,Код_артикул,Формат_кода,"Дата_печати<0x0A><0x0A>На▁изображении▁видно,▁ч...",<0x0A>Продукция_название<>Цена_без_карты<>Цена...
61,25_12-20.mp4,263,1,1043.3,0.845,<0x0A>Продукция_название,NaN,NaN,NaN,Штрихкод,...,NaN,NaN,Цена_без_карты,Цена_с_картой,Скидка_размер,Вид_акции,Код_артикул,Формат_кода,Дата_печати<0x0A><0x0A>Внимательно▁изучите▁изо...,<0x0A>Продукция_название<>Цена_без_карты<>Цена...


{'filename': {0: '25_12-20.mp4',
  1: '25_12-20.mp4',
  2: '25_12-20.mp4',
  3: '25_12-20.mp4',
  4: '25_12-20.mp4',
  5: '25_12-20.mp4',
  6: '25_12-20.mp4',
  7: '25_12-20.mp4',
  8: '25_12-20.mp4',
  9: '25_12-20.mp4',
  10: '25_12-20.mp4',
  11: '25_12-20.mp4',
  12: '25_12-20.mp4',
  13: '25_12-20.mp4',
  14: '25_12-20.mp4',
  15: '25_12-20.mp4',
  16: '25_12-20.mp4',
  17: '25_12-20.mp4',
  18: '25_12-20.mp4',
  19: '25_12-20.mp4',
  20: '25_12-20.mp4',
  21: '25_12-20.mp4',
  22: '25_12-20.mp4',
  23: '25_12-20.mp4',
  24: '25_12-20.mp4',
  25: '25_12-20.mp4',
  26: '25_12-20.mp4',
  27: '25_12-20.mp4',
  28: '25_12-20.mp4',
  29: '25_12-20.mp4',
  30: '25_12-20.mp4',
  31: '25_12-20.mp4',
  32: '25_12-20.mp4',
  33: '25_12-20.mp4',
  34: '25_12-20.mp4',
  35: '25_12-20.mp4',
  36: '25_12-20.mp4',
  37: '25_12-20.mp4',
  38: '25_12-20.mp4',
  39: '25_12-20.mp4',
  40: '25_12-20.mp4',
  41: '25_12-20.mp4',
  42: '25_12-20.mp4',
  43: '25_12-20.mp4',
  44: '25_12-20.mp4',
  45: '2

In [20]:
# Матчинг продуктов с базой данных
df_vlm_match = df_vlm.rename(columns={'product_name': 'ocr_text'})
df_vlm_match = find_top5_matches(df_vlm_match, ocr_col='ocr_text')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4130.69it/s]


In [ ]:
# === Очистка OCR текста от мусора ===
import re

def clean_ocr_text(text: str) -> str:
    """
    Очистка OCR текста от мусора.
    
    Удаляет:
    - XML-подобные теги: <0x0A>, <F0>...
    - Специальные символы: _, ▁, emoji
    - Технические символы: <, >, hex-коды
    
    Оставляет:
    - Буквы (русские + латиница)
    - Цифры
    - Проценты, точки, запятые, дефисы (для объемов: 0.5л, 3.2%)
    """
    if pd.isna(text) or text is None:
        return ""
    
    text = str(text)
    
    # Удаление XML-тегов и hex-кодов
    text = re.sub(r'<[^>]*>', ' ', text)
    text = re.sub(r'0x[0-9A-Fa-f]+', ' ', text)
    
    # Удаление специальных символов
    text = text.replace('_', ' ')
    text = text.replace('▁', ' ')
    
    # Удаление emoji и unicode символов (оставляем буквы, цифры, %, ., ,, -)
    text = re.sub(r'[^\w\sа-яА-Яa-zA-Z0-9.,%-]', ' ', text)
    
    # Удаление множественных пробелов
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Применение очистки к raw_text
df_vlm['raw_text_clean'] = df_vlm['raw_text'].apply(clean_ocr_text)

# Показать примеры очистки
print("=== Примеры очистки OCR текста ===")
for i in range(min(5, len(df_vlm))):
    orig = str(df_vlm.iloc[i]['raw_text'])[:80]
    clean = df_vlm.iloc[i]['raw_text_clean']
    print(f"\n{i+1}. Оригинал: {orig}...")
    print(f"   Очищено:  {clean}")

In [21]:
# === Hugging Face LLM для финального выбора SKU ===
# Использует Qwen2.5-7B-Instruct с 4-bit квантованием
# Работает с очищенным текстом (raw_text_clean)
from LLMTEXT.hf_sku_matcher import HFSKUMatcher

# Инициализация matcher (первый запуск скачает модель ~5GB)
hf_matcher = HFSKUMatcher(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    batch_size=8,  # Пакетная обработка
    max_new_tokens=64,
    temperature=0.1,
    log_to_file=False  # Только консоль
)

# Подготовка данных с очищенным текстом
df_vlm_match = df_vlm.rename(columns={'raw_text_clean': 'ocr_text'})
df_vlm_match = find_top5_matches(df_vlm_match, ocr_col='ocr_text')

# Обработка dataframe с топ-5 кандидатами
# LLM будет искать концептуальные совпадения (не точные)
df_final = hf_matcher.process_dataframe(
    df_vlm_match,
    ocr_col='ocr_text',
    output_col='id_sku'
)

# Освобождение VRAM после обработки
hf_matcher.unload_model()

print(f"\nОбработка завершена! Найдено SKU: {df_final['id_sku'].notna().sum()}/{len(df_final)}")
print(f"Match rate: {df_final['id_sku'].notna().sum() / len(df_final) * 100:.1f}%")

# Показать несколько примеров
print("\n=== Примеры результатов ===")
for i in range(min(5, len(df_final))):
    row = df_final.iloc[i]
    print(f"\n{i+1}. OCR: {row['ocr_text'][:60]}...")
    print(f"   Top1: {row.get('top1', 'N/A')}")
    print(f"   Top1 SKU: {row.get('top1_sku', 'N/A')}")
    print(f"   LLM выбрал: {row['id_sku']}")
    print(f"   Размышления: {str(row.get('llm_thinking', 'N/A'))[:100]}...")

2026-05-19 13:53:59 [INFO] Загрузка модели: Qwen/Qwen2.5-7B-Instruct
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W0519 13:54:02.222000 12032 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   1%|          | 2/339 [00:00<01:29,  3.76it/s]c:\Users\GGamers\Desktop\FLC\hackhatons\lenta\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 339/339 [00:03<00:00, 89.25it/s] 
2026-05-19 13:54:09 [INFO] Модель загружена: Qwen/Qwen2.5-7B-Instruct
2026-05-19 13:54:09 [INFO] Устройство: cuda:0
2026-05-19 13:54:09 [INFO] Начало обработки 63 строк
Обработка: 100%|██████████| 8/8 [00:34<00:00,  4.36s/it]
2026-05-19 13:54:44 [INFO] Обработка завершена. Найдено SKU: 0/63
2026-05-19 13:54:4


Обработка завершена! Найдено SKU: 0/63


In [ ]:
import gc, torch
del yolo, classifier
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()